<a href="https://colab.research.google.com/github/cara-jvr/mit-805-group-project/blob/main/notebooks/Group12_01_Data_exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from pyspark.sql import SparkSession

# creating a Spark Session Object
spark = (
    SparkSession.builder
    .appName("Big-Data-Project")
    .master("local[*]")
    .getOrCreate()
)
sc = spark.sparkContext
spark

In [ ]:
# import libraries
from pyspark.sql.functions import split, explode, col, desc, sum as _sum
import pyarrow.parquet as pq
import glob
import pandas as pd
import os

In [ ]:
# importing raw files for NYC High Volume FHV Trip Data (available at: https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page)
# = ["fhvhv_tripdata_2023-01.parquet",
#              "fhvhv_tripdata_2023-02.parquet",
#             "fhvhv_tripdata_2023-03.parquet",
#             "fhvhv_tripdata_2023-04.parquet",
#              "fhvhv_tripdata_2023-05.parquet",
#              "fhvhv_tripdata_2023-06.parquet"]
#df = spark.read.parquet(*list_files)

# reading files from Google Drive
file_path = "drive/MyDrive/Big_data_project/data"
os.listdir(file_path)

['fhvhv_tripdata_2023-01.parquet',
 'fhvhv_tripdata_2023-02.parquet',
 'fhvhv_tripdata_2023-03.parquet',
 'fhvhv_tripdata_2023-04.parquet',
 'fhvhv_tripdata_2023-05.parquet',
 'fhvhv_tripdata_2023-06.parquet',
 'fhvhv_tripdata_2023-08.parquet',
 'fhvhv_tripdata_2023-09.parquet',
 'fhvhv_tripdata_2023-11.parquet',
 'fhvhv_tripdata_2023-12.parquet',
 'fhvhv_tripdata_2023-07.parquet',
 'fhvhv_tripdata_2023-10.parquet']

In [ ]:
df = spark.read.parquet(file_path)
df.printSchema()

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- originating_base_num: string (nullable = true)
 |-- request_datetime: timestamp_ntz (nullable = true)
 |-- on_scene_datetime: timestamp_ntz (nullable = true)
 |-- pickup_datetime: timestamp_ntz (nullable = true)
 |-- dropoff_datetime: timestamp_ntz (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- trip_miles: double (nullable = true)
 |-- trip_time: long (nullable = true)
 |-- base_passenger_fare: double (nullable = true)
 |-- tolls: double (nullable = true)
 |-- bcf: double (nullable = true)
 |-- sales_tax: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- tips: double (nullable = true)
 |-- driver_pay: double (nullable = true)
 |-- shared_request_flag: string (nullable = true)
 |-- shared_match_flag: string (nullable = true)
 |-- access_a_ride_f

In [ ]:
# number of records
df.count()

232490020

In [ ]:
# size of uncompressed raw file
compressed_gb = 0

for root, dirs, files in os.walk(file_path):
    for file in files:
        if file.endswith(".parquet"):
            compressed_gb += os.path.getsize(os.path.join(root, file))

total_gb = compressed_gb / (1024**3)

print(f"Total dataset size: {total_gb:.2f} GB")

Total dataset size: 5.42 GB


In [ ]:
def get_parquet_uncompressed_size(file_path):
    """Extracts the total uncompressed size from a Parquet file footer metadata."""
    try:
        # Read only the metadata footer, avoiding loading the actual data into memory
        metadata = pq.read_metadata(file_path)

        # Sum up the uncompressed size of all row groups
        total_bytes = sum(metadata.row_group(i).total_byte_size for i in range(metadata.num_row_groups))
        return total_bytes
    except Exception as e:
        print(f"Error reading metadata for {file_path}: {e}")
        return 0

total_uncompressed_bytes = 0
file_count = 0

# Recursively walk through the directory tree
for root, dirs, files in os.walk(file_path):
    for file in files:
        if file.endswith('.parquet') or file.endswith('.parq'):
            full_path = os.path.join(root, file)

            # Get the uncompressed size in bytes
            uncompressed_bytes = get_parquet_uncompressed_size(full_path)
            total_uncompressed_bytes += uncompressed_bytes
            file_count += 1

            # Convert bytes to Gigabytes (GB)
            size_in_gb = uncompressed_bytes / (1024 ** 3)

            # Print individual file metrics (truncated names for alignment)
            display_name = file if len(file) <= 47 else f"...{file[-44:]}"
            print(f"{display_name:<50} | {size_in_gb:>21.4f} GB")

# Final summary conversion
total_uncompressed_gb = total_uncompressed_bytes / (1024 ** 3)

print("=" * 75)
print(f"Total Parquet Files Found: {file_count}")
print(f"Total Uncompressed Size:   {total_uncompressed_gb:.4f} GB")


fhvhv_tripdata_2023-01.parquet                     |                0.7929 GB
fhvhv_tripdata_2023-02.parquet                     |                0.7651 GB
fhvhv_tripdata_2023-03.parquet                     |                0.8706 GB
fhvhv_tripdata_2023-04.parquet                     |                0.8126 GB
fhvhv_tripdata_2023-05.parquet                     |                0.8499 GB
fhvhv_tripdata_2023-06.parquet                     |                0.8261 GB
fhvhv_tripdata_2023-08.parquet                     |                0.5358 GB
fhvhv_tripdata_2023-09.parquet                     |                0.5454 GB
fhvhv_tripdata_2023-11.parquet                     |                0.5366 GB
fhvhv_tripdata_2023-12.parquet                     |                0.5570 GB
fhvhv_tripdata_2023-07.parquet                     |                0.8181 GB
fhvhv_tripdata_2023-10.parquet                     |                0.5614 GB
Total Parquet Files Found: 12
Total Uncompressed Size:   8.4715 

In [ ]:
# Data Cleaning

In [ ]:
# Feature Engineering

In [ ]:
# EDA

## 1.Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip install pyspark -q

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Big_data_project") \
    .getOrCreate()

spark

## 2. Load Files

In [ ]:
from pyspark.sql import functions as F

file_path = "drive/MyDrive/Big_data_project/Data"

df = spark.read.parquet(file_path)


print("Rows:", df.count())
print("Columns:", len(df.columns))
df.printSchema()

## 3. Dataset Characteristics

In [ ]:
df.describe().show()

In [ ]:
df.limit(5).toPandas()

In [ ]:
#Schema
df.printSchema()

In [ ]:
#Data date range
df.select(
    F.min("pickup_datetime"),
    F.max("pickup_datetime")
).show()

In [ ]:
#Dataset Size
import os

total_size = 0

for file in os.listdir("/content"):
    if file.endswith(".parquet"):
        total_size += os.path.getsize("/content/" + file)

print(f"Dataset size: {total_size/(1024**3):.2f} GB")

In [ ]:
#Records per year
df.groupBy(F.year("pickup_datetime").alias("Year")) \
  .count() \
  .orderBy("Year") \
  .show()

## 4. Data Quality Analysis

In [ ]:
#Missing Values
from pyspark.sql.functions import col, when, count

missing = df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
])

missing.show()

In [ ]:
#Duplicate Records
duplicates = df.count() - df.dropDuplicates().count()
print("Duplicate Records:", duplicates)

In [ ]:
#invalid Trip duration
df = df.withColumn(
    "trip_minutes",
    (F.col("dropoff_datetime").cast("long") -
     F.col("pickup_datetime").cast("long"))/60
)

df.filter(F.col("trip_minutes") < 0).count()

## 5. Exploratory Data Analysis (EDA)

In [ ]:
#Total trips
total_trips = df.count()
print(total_trips)

In [ ]:
#Trips by Month
monthly = df.groupBy(
    F.year("pickup_datetime").alias("Year"),
    F.month("pickup_datetime").alias("Month")
).count().orderBy("Year","Month")

monthly.show()

In [ ]:
#PLot
monthly_pd = monthly.toPandas()

import matplotlib.pyplot as plt

plt.figure(figsize=(12,5))
plt.plot(range(len(monthly_pd)), monthly_pd["count"])
plt.title("Trips Per Month")
plt.ylabel("Trips")
plt.show()

In [ ]:
#Trips per year

from pyspark.sql.functions import year

yearly = df.groupBy(
    year("pickup_datetime").alias("Year")
).count()

yearly.show()

In [ ]:
#PLot
yearly_pd = yearly.toPandas()

import matplotlib.pyplot as plt

plt.figure(figsize=(12,5))
plt.plot(range(len(yearly_pd)), yearly_pd["count"])
plt.title("Trips Per Month")
plt.ylabel("Trips")
plt.show()

In [ ]:
#Average Trip Distance
df.select(
    F.avg("trip_miles").alias("Average Distance")
).show()

In [ ]:
#Average Fare
df.select(
    F.avg("base_passenger_fare").alias("Average Fare")
).show()

In [ ]:
#Top Pickup Zone
df.groupBy("PULocationID") \
  .count() \
  .orderBy(F.desc("count")) \
  .show(10)

In [ ]:
#Top Drop off zones

df.groupBy("DOLocationID") \
  .count() \
  .orderBy("count", ascending=False) \
  .show(10)

In [ ]:
#Trips by Hour
hourly = df.groupBy(
    F.hour("pickup_datetime").alias("Hour")
).count().orderBy("Hour")

hourly.show()

In [ ]:
#Plot
hour_pd = hourly.toPandas()

plt.figure(figsize=(10,5))
plt.bar(hour_pd["Hour"], hour_pd["count"])
plt.title("Trips by Hour")
plt.show()